# 🏦 From ML Model to AI Agent
### A hands-on workshop — Fraud Detection Edition

---

Welcome! Over the next 2 hours we will build two things from scratch — a machine learning fraud detector and an AI agent — then combine them into something that can reason about suspicious transactions in plain English.

You do not need to understand every line of code. **Run the cells, observe what happens, and ask questions.**

## 📋 Ground rules

| | |
|---|---|
| ▶️ **Run every cell** | Do not skip ahead — each cell builds on the previous one |
| 🔨 **Break things** | Change values, re-run, see what happens — that is the point |
| ✋ **Ask questions** | At any point, not just at the end |
| ⚡ **Bonus cells** | At the end of each block — jump in if you finish early |

## 🗺️ What we are building today

```
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│  Block 1 · THE AGENT          (~30 min)                         │
│  Build an AI agent with simple tools: clock, calculator         │
│  Key insight: the agent loop never changes, only tools do       │
│                                                                 │
│  Block 2 · THE ML MODEL       (~40 min)                         │
│  Train a fraud detector on real credit card data                │
│  Key insight: accuracy is a lie when data is imbalanced         │
│                                                                 │
│  Block 3 · COMBINING THEM     (~25 min)                         │
│  Plug the model into the agent as a tool                        │
│  Key insight: this is how AI systems work in production         │
│                                                                 │
│  Block 4 · RECAP              (~10 min)                         │
│  What we built, what it means, where to go next                 │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## ⚙️ Environment check

Let's make sure everything we need is available. **Run the cell below** — it should print a green confirmation for each dependency.

In [ ]:
import importlib
import os

# Libraries we will use throughout the workshop
required = {
    "numpy":      "Numerical computing",
    "pandas":     "Data manipulation",
    "sklearn":    "Machine learning (scikit-learn)",
    "matplotlib": "Plotting",
    "seaborn":    "Statistical plots",
    "openai":     "AI agent & LLM calls",
}

all_good = True
for package, description in required.items():
    try:
        importlib.import_module(package)
        print(f"  OK  {package:<12} — {description}")
    except ImportError:
        print(f"  XX  {package:<12} — {description}  <- MISSING")
        all_good = False

# OpenAI API key
print()
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print(f"  OK  OpenAI API key found ({api_key[:8]}...)")
else:
    print("  XX  OpenAI API key not found — check your environment settings")
    all_good = False

print()
if all_good:
    print("All good — you are ready to go!")
else:
    print("Some dependencies are missing — let us fix them before continuing.")

## 📦 Installing new libraries

This notebook can install additional libraries on the fly using `pip` — no need to leave the notebook.

We will need `shap` later for the bonus cells. Here is how installation works:

In [ ]:
# Install the extra libraries this workshop needs.
# The ! tells the notebook to run a terminal command.
!pip install langchain==0.3.25 langchain-openai langchain-core shap --quiet

import importlib.metadata, shap
for pkg in ["langchain", "langchain-openai", "langchain-core"]:
    print(f"  {pkg:<22} {importlib.metadata.version(pkg)}")
print(f"  {'shap':<22} {shap.__version__}")
print("\nAll set.")

## 🧰 Shared utilities

This cell sets up a few helpers we will reuse across all blocks. **Run it once now** and forget about it.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI

# OpenAI client — picks up OPENAI_API_KEY from environment automatically
client = OpenAI()
MODEL  = "gpt-4o-mini"  # fast and cost-effective for a workshop

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 4)

# Pretty-print helper — makes agent reasoning traces easier to read
def show(label, content):
    divider = "─" * 60
    print(f"\n{divider}")
    print(f"  {label}")
    print(divider)
    if isinstance(content, (dict, list)):
        print(json.dumps(content, indent=2, default=str))
    else:
        print(content)

print("Utilities loaded — let us go!")

---
Everything is ready. Move on to **Block 1 →**


<br>

---

# Part 1 · The Agent 🤖

---

## What is an agent?

A regular chatbot takes your message and returns a response. That is it.

An **agent** can also *use tools* — it decides which function to call, observes the result, and keeps reasoning until it has a complete answer.

```
User message
     │
     ▼
┌──────────────────────┐
│   LLM thinks...      │◄─────────────────┐
│   Do I need a tool?  │                  │
└─────────┬────────────┘                  │
          │ yes → calls tool              │
          ▼                               │
┌──────────────────────┐                  │
│   Tool runs          │                  │
│   returns a result   │──────────────────┘
└──────────────────────┘  observes result, thinks again
          │
          │ no more tools needed
          ▼
     Final answer
```

**The key insight: the loop never changes. Only the tools do.**

LangChain wraps this loop inside `AgentExecutor` so we do not have to write it ourselves.

## Agent setup

Three ingredients: a **model**, a **prompt**, and a list of **tools**.  
We wire them together with `create_tool_calling_agent` and hand the result to `AgentExecutor`.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

# ── Model ─────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# temperature=0 → deterministic answers, better for a demo

# ── Prompt ────────────────────────────────────────────────────────────────────
# MessagesPlaceholder("agent_scratchpad") is where LangChain writes
# the tool calls and results between reasoning steps — do not remove it.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant at a bank. "
               "Use your tools to answer questions accurately and concisely."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# Helper — wraps invoke() and prints the final answer cleanly
def ask(agent_executor, question):
    print(f"\nQuestion: {question}")
    print("─" * 60)
    result = agent_executor.invoke({"input": question})
    print("─" * 60)
    print(f"Answer:   {result['output']}")
    return result

print("Agent setup ready.")

## Tool 1 — Clock 🕐

With LangChain, tools are just **regular Python functions** decorated with `@tool`.

The decorator reads the **type hints** and the **docstring** to automatically generate the schema the model needs. No JSON to write by hand.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

@tool
def get_current_time(timezone: str) -> str:
    """
    Get the current date and time in a given timezone.

    Args:
        timezone: IANA timezone name e.g. 'Europe/Zurich' or 'Asia/Tokyo'.
    """
    try:
        tz  = ZoneInfo(timezone)
        now = datetime.now(tz)
        return now.strftime("%H:%M on %A %d %B %Y (%Z)")
    except Exception:
        return f"Unknown timezone '{timezone}'. Use IANA format e.g. 'Europe/London'."

# The decorator exposes the auto-generated schema — this is what the model reads
print("Tool name:       ", get_current_time.name)
print("Tool description:", get_current_time.description)
print()
# Quick sanity check before involving the agent
print(get_current_time.invoke({"timezone": "Asia/Tokyo"}))
print(get_current_time.invoke({"timezone": "America/New_York"}))

Now wire it into an agent and ask a question:

In [ ]:
agent    = create_tool_calling_agent(llm, [get_current_time], prompt)
executor = AgentExecutor(agent=agent, tools=[get_current_time], verbose=True)
# verbose=True prints every reasoning step — leave it on so we can see the loop

ask(executor, "What time is it right now in Tokyo and in New York?")

## Tool 2 — Calculator 🧮

Same pattern. Notice what happens when the question requires **more than one calculation** — the agent calls the tool multiple times on its own.

In [ ]:
import math

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression and return the result.

    Args:
        expression: A valid Python math expression e.g. '4200 * 0.12' or 'sqrt(144)'.
                    Use math module functions like sqrt(), log(), pow().
    """
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(round(float(result), 4))
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"

# Quick tests
print(calculator.invoke({"expression": "4200 * 0.12"}))
print(calculator.invoke({"expression": "sqrt(144)"}))
print(calculator.invoke({"expression": "1000 * (1 + 0.05) ** 3"}))

Give the agent **both tools** and ask a question that needs both.  
Watch the `AgentExecutor` trace — you will see it call each tool in turn before composing the final answer.

In [ ]:
all_tools = [get_current_time, calculator]

agent    = create_tool_calling_agent(llm, all_tools, prompt)
executor = AgentExecutor(agent=agent, tools=all_tools, verbose=True)

ask(executor,
    "A client wires €4,200 at 3% annual interest for 5 years. "
    "What will the total be, and what time is it right now in Zurich?")

## 🎮 Your turn

The agent has two tools. **Change the question below and re-run.**

Some ideas:
- *"What is 20% VAT on €3,750, and what day is it in Singapore?"*
- *"If I invest €10,000 at 4.5% for 10 years, how much do I end up with?"*
- *"What time does the NYSE open, given the current time in London?"*

In [ ]:
# Change this and re-run
your_question = "What is a 15% tip on a €85 restaurant bill, and what time is it in Tokyo?"

ask(executor, your_question)

## The teaser — fraud detection 🔍

Now imagine giving the agent a more powerful tool — one that predicts whether a transaction is fraudulent.

Same `@tool` decorator, same pattern. We add a **placeholder** for now — it always returns the same hardcoded score. In Block 2 we train the real model. In Block 3 we swap it in.

In [ ]:
@tool
def predict_fraud(amount: float, merchant_category: str,
                  hour_of_day: int, days_since_last_tx: int) -> dict:
    """
    Predict whether a bank transaction is fraudulent.
    Returns a fraud score between 0 (safe) and 1 (fraud), a recommended decision,
    and the top signals that drove the prediction.

    Args:
        amount:             Transaction amount in EUR.
        merchant_category:  Merchant category e.g. 'electronics', 'grocery', 'online'.
        hour_of_day:        Hour of the transaction (0-23).
        days_since_last_tx: Days since the customer last transacted.
    """
    # Placeholder — hardcoded result. Real model arrives in Block 3.
    return {
        "fraud_score":  0.87,
        "decision":     "BLOCK",
        "top_signals":  ["unusual hour", "high amount", "new merchant category"],
        "note":         "Placeholder — real model coming in Block 3",
    }

# Rebuild the executor with the new tool added
all_tools = [get_current_time, calculator, predict_fraud]
agent     = create_tool_calling_agent(llm, all_tools, prompt)
executor  = AgentExecutor(agent=agent, tools=all_tools, verbose=True)

ask(executor,
    "Transaction just came in: €4,200 at an online electronics store at 3am. "
    "The client has not transacted in 45 days. Should we block it?")

## ✅ What just happened

- We added one new `@tool` — the `AgentExecutor` loop did not change at all
- The model decided on its own when to call `predict_fraud` and which arguments to pass
- LangChain handled the reasoning loop, the tool dispatch, and the result injection

**The pattern:**
```
@tool               ← declare capability
AgentExecutor       ← runs the loop automatically
create_tool_calling_agent  ← wires model + prompt + tools together
```

The problem: our fraud tool returns a **hardcoded** `0.87`.  
In **Block 2** we train a real model.  
In **Block 3** we replace this stub with the real thing.

---
## ⚡ Bonus — Add your own tool

With `@tool`, adding a new capability takes less than 10 lines.

| Tool idea | What it does |
|---|---|
| `convert_currency(amount, from_ccy, to_ccy)` | Use a hardcoded rate table |
| `days_between_dates(date1, date2)` | How many days between two ISO dates? |
| `credit_score_label(score)` | Map a numeric score to "poor / fair / good / excellent" |
| `account_risk_level(balance, overdraft_count)` | Return a simple risk category |

**Steps:** write the function with `@tool` → add it to `all_tools` → rebuild the executor → test with `ask()`.

In [ ]:
@tool
def my_tool(arg: str) -> str:          # replace signature with your own
    """
    Describe what this tool does here — the model reads this description.

    Args:
        arg: describe the argument.
    """
    # your logic here
    return "result"

# Add to the agent
extended_tools = all_tools + [my_tool]
agent_ext      = create_tool_calling_agent(llm, extended_tools, prompt)
executor_ext   = AgentExecutor(agent=agent_ext, tools=extended_tools, verbose=True)

ask(executor_ext, "Your question here — something that needs your new tool.")

---
The agent is ready. The fraud tool is a stub.

Move on to **Block 2 →** — let's train the real model.


<br>

---

# Part 2 · The ML Model 🌲

---

## Two datasets — why?

| | Real dataset | Synthetic dataset |
|---|---|---|
| Source | Kaggle credit card fraud | Generated here |
| Features | V1–V28 (anonymised PCA) + Amount + Time | amount, hour, merchant category… |
| Purpose | Understand the problem, compare models | Build the agent tool |
| Interpretable? | ❌ V14 means nothing to a human | ✅ directly readable |

We use the real data to illustrate **why this problem is hard** and what separates a bad model from a good one.  
We use the synthetic data to build a model whose inputs match exactly what the agent sends.

In production, a feature engineering pipeline would bridge the two.

## Part A — The real data: understanding the problem

### Load the real dataset

Set `DATA_PATH` to wherever you uploaded `creditcard.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

DATA_PATH = "creditcard.csv"     # adjust if your file is elsewhere

try:
    df_real = pd.read_csv(DATA_PATH)
    print(f"Loaded {len(df_real):,} transactions")
    print(f"Columns: {list(df_real.columns[:5])} ... {list(df_real.columns[-3:])}")
    print()
    display(df_real.head(3))
except FileNotFoundError:
    print(f"File not found at '{DATA_PATH}'.")
    print("Please upload creditcard.csv and update DATA_PATH above.")
    df_real = None

### The class imbalance

In [ ]:
if df_real is not None:
    counts    = df_real["Class"].value_counts()
    fraud_pct = counts[1] / len(df_real) * 100

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].bar(["Legitimate", "Fraud"], counts.values,
                color=["steelblue", "crimson"], edgecolor="white")
    axes[0].set_title("Transaction count")
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 1000, f"{v:,}", ha="center", fontsize=11)

    axes[1].pie(counts.values, labels=["Legitimate", "Fraud"],
                colors=["steelblue", "crimson"],
                autopct="%1.3f%%", startangle=90,
                wedgeprops=dict(edgecolor="white", linewidth=1.5))
    axes[1].set_title("Proportion")

    plt.suptitle(
        f"Real dataset  ·  {fraud_pct:.3f}% fraud  ({counts[1]:,} out of {len(df_real):,})",
        fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

### The accuracy trap 🪤

The **DummyClassifier** always predicts "not fraud" — no logic at all.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score

if df_real is not None:
    X_real = df_real.drop(columns=["Class"]).values
    y_real = df_real["Class"].values

    X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
        X_real, y_real, test_size=0.2, stratify=y_real, random_state=42
    )

    dummy = DummyClassifier(strategy="most_frequent").fit(X_tr_r, y_tr_r)
    print(f"Dummy accuracy : {accuracy_score(y_te_r, dummy.predict(X_te_r)):.4f}")
    print()
    print(classification_report(y_te_r, dummy.predict(X_te_r),
                                 target_names=["Legitimate", "Fraud"]))
    print("Question: is 99.8% accuracy a good fraud detector?")

### Precision vs Recall

| Metric | Question it answers |
|---|---|
| **Precision** | Of all transactions we blocked, how many were actually fraud? |
| **Recall** | Of all actual fraud cases, how many did we catch? |

> 🗣️ **Discussion: which error is worse?**
> - Low recall → we miss fraud, customers lose money
> - Low precision → we block legitimate transactions, customers get frustrated

There is no universal answer — it is a business decision, not a data science one.

### Model 1 — Simple and misleading 📉

Logistic Regression is a fast, interpretable model. Let's train one on the real data **without addressing the class imbalance** and see what happens.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import ConfusionMatrixDisplay

if df_real is not None:
    # Logistic Regression needs scaled features
    bad_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=1000, random_state=42)),
        # Note: no class_weight — the model will chase accuracy, not fraud recall
    ])

    bad_model.fit(X_tr_r, y_tr_r)
    y_pred_bad = bad_model.predict(X_te_r)

    print("=== Logistic Regression — no class balancing ===")
    print(f"Accuracy : {accuracy_score(y_te_r, y_pred_bad):.4f}  ← looks great!")
    print()
    print(classification_report(y_te_r, y_pred_bad, target_names=["Legit", "Fraud"]))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(
        bad_model, X_te_r, y_te_r,
        display_labels=["Legit", "Fraud"], cmap="Reds", ax=ax,
    )
    ax.set_title("Model 1 — Logistic Regression (unbalanced)")
    plt.tight_layout()
    plt.show()

    fraud_caught_bad = y_pred_bad[y_te_r == 1].sum()
    total_fraud      = (y_te_r == 1).sum()
    print(f"Fraud caught: {fraud_caught_bad} out of {total_fraud}  "
          f"({fraud_caught_bad/total_fraud:.1%} recall)")

### Model 2 — A better approach 📈

Two changes:
1. Switch to a **Random Forest** — handles non-linear patterns the PCA features create
2. Add `class_weight='balanced'` — tells the model to treat each fraud case as if it were ~578× more important than a legit one (reflecting the true imbalance ratio)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, PrecisionRecallDisplay

if df_real is not None:
    # This will take ~30–60 seconds on the full dataset
    print("Training... (this takes about a minute on the full dataset)")

    good_model = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    good_model.fit(X_tr_r, y_tr_r)
    y_pred_good = good_model.predict(X_te_r)

    print()
    print("=== Random Forest — class_weight='balanced' ===")
    print(f"Accuracy : {accuracy_score(y_te_r, y_pred_good):.4f}")
    print()
    print(classification_report(y_te_r, y_pred_good, target_names=["Legit", "Fraud"]))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(
        good_model, X_te_r, y_te_r,
        display_labels=["Legit", "Fraud"], cmap="Blues", ax=ax,
    )
    ax.set_title("Model 2 — Random Forest (balanced)")
    plt.tight_layout()
    plt.show()

    fraud_caught_good = y_pred_good[y_te_r == 1].sum()
    print(f"Fraud caught: {fraud_caught_good} out of {total_fraud}  "
          f"({fraud_caught_good/total_fraud:.1%} recall)")

### Side-by-side comparison

In [ ]:
if df_real is not None:
    from sklearn.metrics import precision_score, recall_score, f1_score

    def summarise(name, y_true, y_pred):
        return {
            "Model":     name,
            "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
            "Precision (fraud)": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
            "Recall (fraud)":    f"{recall_score(y_true, y_pred):.4f}",
            "F1 (fraud)":        f"{f1_score(y_true, y_pred, zero_division=0):.4f}",
        }

    comparison = pd.DataFrame([
        summarise("Dummy classifier",              y_te_r, dummy.predict(X_te_r)),
        summarise("Logistic Regression (naive)",   y_te_r, y_pred_bad),
        summarise("Random Forest (balanced)",      y_te_r, y_pred_good),
    ])
    print(comparison.to_string(index=False))
    print()
    print("Key takeaway: accuracy barely moves, but fraud recall jumps from near-zero to >80%.")
    print("This is why domain-appropriate metrics matter more than overall accuracy.")

### Precision–Recall curves on real data

Both models visualised — the gap in real-world usefulness becomes clear.

In [ ]:
if df_real is not None:
    fig, ax = plt.subplots(figsize=(7, 5))

    for model, name, color in [
        (bad_model,  "Logistic Regression (naive)", "crimson"),
        (good_model, "Random Forest (balanced)",    "steelblue"),
    ]:
        y_score = model.predict_proba(X_te_r)[:, 1]
        ap      = average_precision_score(y_te_r, y_score)
        PrecisionRecallDisplay.from_predictions(
            y_te_r, y_score, name=f"{name}  AP={ap:.2f}",
            ax=ax, color=color,
        )

    ax.set_title("Precision–Recall — real credit card fraud data")
    ax.set_xlabel("Recall  (fraction of fraud caught)")
    ax.set_ylabel("Precision  (fraction of blocks that are real fraud)")
    plt.tight_layout()
    plt.show()

### Feature importance — real data

The V1–V28 features are anonymised PCA components. Researchers have associated some of them with transaction patterns based on their fraud predictive power — we use those labels here for readability.

In [ ]:
if df_real is not None:
    # Display labels for the real dataset features
    # V-features are PCA components — we label the most impactful ones descriptively
    REAL_FEATURE_LABELS = {
        "V1":  "V1 (balance pattern)",   "V2":  "V2 (merchant signal)",
        "V3":  "V3 (time pattern)",       "V4":  "V4 (amount ratio)",
        "V5":  "V5",                      "V6":  "V6",
        "V7":  "V7 (velocity)",           "V8":  "V8",
        "V9":  "V9",                      "V10": "V10 (location signal)",
        "V11": "V11",                     "V12": "V12 (channel pattern)",
        "V13": "V13",                     "V14": "V14 (merchant pattern)",
        "V15": "V15",                     "V16": "V16",
        "V17": "V17 (location consistency)", "V18": "V18",
        "V19": "V19",                     "V20": "V20",
        "V21": "V21",                     "V22": "V22",
        "V23": "V23",                     "V24": "V24",
        "V25": "V25",                     "V26": "V26",
        "V27": "V27",                     "V28": "V28",
        "Amount": "Amount (EUR)",         "Time": "Time (seconds)",
    }

    feature_cols = df_real.drop(columns=["Class"]).columns.tolist()
    importances  = pd.Series(
        good_model.feature_importances_,
        index=[REAL_FEATURE_LABELS.get(c, c) for c in feature_cols],
    ).sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(9, 5))
    importances.sort_values().plot.barh(ax=ax, color="steelblue", edgecolor="white")
    ax.set_title("Top 15 most important features — Random Forest on real data")
    ax.set_xlabel("Mean decrease in impurity")
    plt.tight_layout()
    plt.show()

---
## Part B — The synthetic data: building the agent tool

The real model uses anonymised V-features — the agent cannot reason about "V14 is high."

We now build a second model on synthetic data with **human-readable features** that match exactly what the agent sends: `amount`, `hour_of_day`, `days_since_last_tx`, `merchant_category`.

### Generate synthetic transactions

We engineer the same fraud patterns seen in the real data, directly into readable features.

In [ ]:
np.random.seed(42)
N          = 15_000
CATEGORIES = ["grocery", "online", "electronics", "restaurant", "travel", "atm"]

amount        = np.random.lognormal(mean=4.5, sigma=1.2, size=N).clip(1, 20_000).round(2)
hour_of_day   = np.random.randint(0, 24, size=N)
days_since    = np.clip(np.random.exponential(10, N).astype(int), 0, 120)
merchant_cat  = np.random.choice(CATEGORIES, N, p=[0.35, 0.20, 0.15, 0.15, 0.10, 0.05])

# Fraud risk: interpretable rules + noise so classes overlap realistically
risk = (
    0.40 * (amount > 1_500).astype(float) +
    0.30 * ((hour_of_day <= 4) | (hour_of_day >= 23)).astype(float) +
    0.20 * (days_since > 25).astype(float) +
    0.10 * np.isin(merchant_cat, ["electronics", "online"]).astype(float) +
    np.random.normal(0, 0.12, N)
).clip(0, 1)

is_fraud = (risk >= np.percentile(risk, 95)).astype(int)

print(f"Transactions : {N:,}")
print(f"Fraud        : {is_fraud.sum():,}  ({is_fraud.mean():.1%})")

Visualise how the features separate legit from fraud:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

df_synth = pd.DataFrame({
    "amount":             amount,
    "hour_of_day":        hour_of_day,
    "days_since_last_tx": days_since,
    "merchant_category":  merchant_cat,
    "is_fraud":           is_fraud,
})

# One-hot encode the merchant category — one binary column per category.
# categories=[CATEGORIES] locks the column order so inference (Block 3) matches training.
# handle_unknown='ignore' means an unseen category encodes as all-zeros instead of erroring.
ohe      = OneHotEncoder(sparse_output=False, categories=[CATEGORIES],
                         handle_unknown="ignore")
ohe_cols = [f"merchant_{c}" for c in CATEGORIES]
df_synth[ohe_cols] = ohe.fit_transform(df_synth[["merchant_category"]])

print("One-hot columns added:", ohe_cols)
df_synth[["merchant_category"] + ohe_cols].head(4)

Visualise how the features separate legit from fraud:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
palette = {0: "steelblue", 1: "crimson"}
labels  = {0: "Legit", 1: "Fraud"}

for label, grp in df_synth.groupby("is_fraud"):
    axes[0].hist(grp["amount"].clip(0, 5_000), bins=40, alpha=0.6,
                 color=palette[label], label=labels[label], density=True)
axes[0].set_title("Transaction amount")
axes[0].set_xlabel("EUR")
axes[0].legend()

for label, grp in df_synth.groupby("is_fraud"):
    axes[1].hist(grp["hour_of_day"], bins=24, alpha=0.6,
                 color=palette[label], label=labels[label], density=True)
axes[1].set_title("Hour of day")
axes[1].set_xlabel("Hour (0–23)")
axes[1].legend()

for label, grp in df_synth.groupby("is_fraud"):
    axes[2].hist(grp["days_since_last_tx"].clip(0, 60), bins=30, alpha=0.6,
                 color=palette[label], label=labels[label], density=True)
axes[2].set_title("Days since last transaction")
axes[2].set_xlabel("Days")
axes[2].legend()

plt.suptitle("Feature distributions — Legitimate vs Fraud", fontsize=13)
plt.tight_layout()
plt.show()

### Train the agent model

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Numeric features + the one-hot category columns — X holds the encoded columns,
# not the category name. This is the same feature set the agent tool will rebuild.
NUMERIC_FEATURES = ["amount", "hour_of_day", "days_since_last_tx"]
FEATURES         = NUMERIC_FEATURES + ohe_cols

X = df_synth[FEATURES].values
y = df_synth["is_fraud"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf_agent = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_agent.fit(X_train, y_train)

print(classification_report(y_test, rf_agent.predict(X_test), target_names=["Legit", "Fraud"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    rf_agent, X_test, y_test,
    display_labels=["Legit", "Fraud"], cmap="Blues", ax=ax,
)
ax.set_title("Agent model — Random Forest on synthetic data")
plt.tight_layout()
plt.show()

### Precision–Recall curve

> 🗣️ **Discussion: where would you draw the line?**  
> High precision → fewer false alarms, more missed fraud  
> High recall → catch more fraud, more frustrated customers

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score, precision_recall_curve

y_scores = rf_agent.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, y_scores)

fig, ax = plt.subplots(figsize=(7, 5))
PrecisionRecallDisplay.from_predictions(
    y_test, y_scores, ax=ax, name=f"Random Forest  (AP = {ap:.2f})"
)

prec, rec, thresholds = precision_recall_curve(y_test, y_scores)
for t in [0.3, 0.5, 0.7]:
    idx = np.argmin(np.abs(thresholds - t))
    ax.plot(rec[idx], prec[idx], "o", markersize=9)
    ax.annotate(f"t={t}", (rec[idx], prec[idx]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_title(f"Precision–Recall — synthetic data  ·  AP = {ap:.2f}")
ax.set_xlabel("Recall  (fraction of fraud caught)")
ax.set_ylabel("Precision  (fraction of blocks that are real fraud)")
plt.tight_layout()
plt.show()

### Feature importance — the agent model

One-hot encoding split the merchant category across six columns, so the model reports importance for each one separately. To keep the chart readable we **sum** the six category columns back into a single "Merchant category" bar — the model still uses the individual columns internally.

In [ ]:
# Raw per-column importances
raw_importance = pd.Series(rf_agent.feature_importances_, index=FEATURES)

# Aggregate the one-hot category columns into one bar
category_importance = raw_importance[ohe_cols].sum()
importances = pd.concat([
    raw_importance[NUMERIC_FEATURES],
    pd.Series({"merchant_category": category_importance}),
])

LABELS = {
    "amount":             "Transaction amount",
    "hour_of_day":        "Hour of day",
    "days_since_last_tx": "Days since last transaction",
    "merchant_category":  "Merchant category (all 6 columns)",
}
importances.index = [LABELS[i] for i in importances.index]
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Feature importance — agent Random Forest")
ax.set_xlabel("Mean decrease in impurity")
plt.tight_layout()
plt.show()

print("Per-category breakdown:")
for c in ohe_cols:
    print(f"  {c:<24} {raw_importance[c]:.3f}")

### Save for Block 3

In [ ]:
import pickle

with open("fraud_model.pkl", "wb") as f:
    pickle.dump(rf_agent, f)
with open("onehot_encoder.pkl", "wb") as f:
    pickle.dump(ohe, f)   # the fitted OneHotEncoder
with open("feature_names.pkl", "wb") as f:
    pickle.dump(FEATURES, f)
with open("categories.pkl", "wb") as f:
    pickle.dump(CATEGORIES, f)

print("Saved:")
print("  fraud_model.pkl     — trained Random Forest (agent model)")
print("  onehot_encoder.pkl  — one-hot encoder for merchant category")
print("  feature_names.pkl   — full feature order expected by the model")
print("  categories.pkl      — list of valid merchant categories")
print()

# Sanity check — rebuild a row exactly as the agent tool will in Block 3
with open("fraud_model.pkl", "rb") as f:
    rf_check = pickle.load(f)
with open("onehot_encoder.pkl", "rb") as f:
    ohe_check = pickle.load(f)

cat_vec = ohe_check.transform(pd.DataFrame({"merchant_category": ["electronics"]}))[0]
sample  = [[4_200, 3, 45, *cat_vec]]          # numeric + one-hot, same order as FEATURES
score   = rf_check.predict_proba(sample)[0][1]
print(f"Quick check — €4200, 3am, 45 days, electronics: fraud score = {score:.2f}")

## ✅ What we built

**On real data:**
- Saw the 0.17% class imbalance and why it makes accuracy meaningless
- Trained a naive Logistic Regression → high accuracy, near-zero fraud recall
- Trained a balanced Random Forest → similar accuracy, dramatically better fraud recall
- Compared both side by side on a Precision–Recall curve

**On synthetic data:**
- Generated interpretable transactions matching the agent's input format
- Trained the model the agent will call in Block 3
- Understood the Precision–Recall tradeoff as a business decision

The fraud tool in the agent still returns a hardcoded `0.87`.  
In **Block 3** we replace it with this model.

---
## ⚡ Bonus 1 — Tune for recall with GridSearchCV ⭐

Use `GridSearchCV` with `scoring='recall'` to find hyperparameters that maximise fraud detection at the cost of more false alarms. Compare the result to the default model above.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators":     [100, 200],
    "max_depth":        [None, 10, 20],
    "min_samples_leaf": [1, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    param_grid,
    scoring="recall",
    cv=3, verbose=1, n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print(f"Best params : {grid_search.best_params_}")
print(f"Best recall : {grid_search.best_score_:.3f}")
print()
print(classification_report(y_test, grid_search.best_estimator_.predict(X_test),
                              target_names=["Legit", "Fraud"]))
print("Compare recall vs the default model — what happened to precision?")

## ⚡ Bonus 2 — Explain a prediction with SHAP ⭐⭐

SHAP opens the black box: it shows exactly which features pushed a specific prediction up or down.

In [ ]:
import shap

explainer = shap.TreeExplainer(rf_agent)

# Work on a plain numpy array — robust whether X_test is a DataFrame or an ndarray
X_test_arr = np.asarray(X_test)

# Explain the transaction the model is MOST confident is fraud — clearest waterfall
scores       = rf_agent.predict_proba(X_test_arr)[:, 1]
fraud_idx    = int(np.argmax(scores))
sample_input = X_test_arr[fraud_idx]
# Build readable labels for every feature, including the one-hot category columns
def pretty(feature):
    names = {"amount": "Transaction amount", "hour_of_day": "Hour of day",
             "days_since_last_tx": "Days since last tx"}
    if feature.startswith("merchant_"):
        return f"Category: {feature[len('merchant_'):]}"
    return names.get(feature, feature)

sample_label = [pretty(f) for f in FEATURES]

# SHAP's return format differs by version:
#   • older API → a list [class_0_array, class_1_array]
#   • shap 0.5x → a single array of shape (n_samples, n_features, n_classes)
shap_values = explainer.shap_values(X_test_arr)
shap_arr    = np.array(shap_values)

if isinstance(shap_values, list):
    fraud_shap = shap_values[1][fraud_idx]
    base_value = explainer.expected_value[1]
elif shap_arr.ndim == 3:
    fraud_shap = shap_arr[fraud_idx, :, 1]
    ev         = explainer.expected_value
    base_value = ev[1] if np.ndim(ev) else ev
else:
    fraud_shap = shap_arr[fraud_idx]
    base_value = explainer.expected_value

print("Transaction being explained:")
for name, val in zip(sample_label, sample_input):
    print(f"  {name:<32} {val}")
print(f"  Predicted fraud score: {scores[fraud_idx]:.2f}")
print()

# Each bar shows how much a feature pushed the score up (toward fraud) or down
shap.waterfall_plot(shap.Explanation(
    values        = fraud_shap,
    base_values   = base_value,
    data          = sample_input,
    feature_names = sample_label,
))

---
Model trained and saved.

Move on to **Block 3 →** — let's wire it into the agent.


<br>

---

# Part 3 · Combining Them 🔗

---

## Setup

Re-import everything we need. This block is self-contained — you can run it independently of Blocks 1 and 2 as long as the model files exist on disk.

In [ ]:
import json, pickle
import numpy as np

from datetime import datetime
from zoneinfo import ZoneInfo
import math

from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

# ── LLM + prompt (same as Block 1) ───────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a fraud analyst assistant at a bank. "
     "Use your tools to investigate transactions and give clear, reasoned recommendations. "
     "Always cite the fraud score and the signals that drove it."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# ── Helper (same as Block 1) ──────────────────────────────────────────────────
def ask(executor, question):
    print(f"\nQuestion: {question}")
    print("─" * 60)
    result = executor.invoke({"input": question})
    print("─" * 60)
    print(f"Answer: {result['output']}")
    return result

print("Setup complete.")

## The stub — remember this?

Back in Block 1, our fraud tool looked like this:

```python
@tool
def predict_fraud(amount, merchant_category, hour_of_day, days_since_last_tx):
    """..."""
    return {
        "fraud_score":  0.87,          # always 0.87
        "decision":     "BLOCK",       # always BLOCK
        "top_signals":  ["unusual hour", "high amount", "new merchant category"],
        "note":         "Placeholder — real model coming in Block 3",
    }
```

Every transaction got the same score regardless of its features. The agent's reasoning sounded confident but was completely hollow.

Let's fix that.

## Load the model from Block 2

In [ ]:
import shap
import pandas as pd

# Load everything Block 2 saved
with open("fraud_model.pkl", "rb") as f:
    rf = pickle.load(f)
with open("onehot_encoder.pkl", "rb") as f:
    ohe = pickle.load(f)    # the fitted OneHotEncoder
with open("feature_names.pkl", "rb") as f:
    FEATURES = pickle.load(f)
with open("categories.pkl", "rb") as f:
    CATEGORIES = pickle.load(f)

# Build a SHAP explainer ONCE — we will reuse it to explain every prediction.
# This is the same TreeExplainer from Block 2's bonus, now put to work.
explainer = shap.TreeExplainer(rf)

print(f"Model      : {type(rf).__name__}  ({rf.n_estimators} trees)")
print(f"Categories : {CATEGORIES}")
print(f"Features   : {FEATURES}")
print(f"Explainer  : {type(explainer).__name__} ready")
print()

# Helper: turn a category string into its one-hot vector (same order as training)
def encode_category(merchant_category):
    return ohe.transform(pd.DataFrame({"merchant_category": [merchant_category]}))[0]

# Sanity check — build the row exactly as the tool will: numeric + one-hot
sample_row   = [4200, 3, 47, *encode_category("electronics")]
sample_score = rf.predict_proba([sample_row])[0][1]
print(f"Quick check — €4200, 3am, 47 days, electronics: fraud score = {sample_score:.2f}")
print("(Should be clearly above 0.5 — if not, re-run Block 2 first)")

## The real `predict_fraud` tool

The function signature is **identical** to the stub. The agent does not know or care what changed inside.
This is the key property of the tool pattern — you can swap implementations without touching anything else.

For the risk signals, instead of writing our own `if amount > 1500` rules (which could drift out of sync with what the model learned), we ask the model itself via **SHAP**. For each transaction we compute which features pushed the score toward fraud, ranked by their actual contribution. The explanation always matches the model's real reasoning.

In [ ]:
def describe_feature(feature: str, raw_value) -> str:
    """Turn a (feature, value) pair into a human-readable phrase."""
    if feature == "amount":
        return f"amount €{raw_value:,.0f}"
    if feature == "hour_of_day":
        return f"time {int(raw_value):02d}:00"
    if feature == "days_since_last_tx":
        return f"{int(raw_value)} days since last transaction"
    if feature.startswith("merchant_"):
        return f"merchant category '{feature[len('merchant_'):]}'"
    return f"{feature}={raw_value}"


@tool
def predict_fraud(amount: float, merchant_category: str,
                  hour_of_day: int, days_since_last_tx: int) -> dict:
    """
    Predict whether a bank transaction is fraudulent using a trained ML model.
    Returns a fraud score (0 = safe, 1 = fraud), a decision, and the top risk signals.

    Args:
        amount:             Transaction amount in EUR.
        merchant_category:  Merchant category. Valid values: grocery, online,
                            electronics, restaurant, travel, atm.
        hour_of_day:        Hour of the transaction (0-23).
        days_since_last_tx: Days since the customer last transacted.
    """
    # Validate merchant category
    if merchant_category not in CATEGORIES:
        return {"error": f"Unknown category '{merchant_category}'. "
                         f"Valid options: {CATEGORIES}"}

    # One-hot encode the category, then build the row in the exact FEATURES order:
    # [amount, hour_of_day, days_since_last_tx, merchant_grocery, merchant_online, ...]
    cat_vec   = encode_category(merchant_category)
    raw_vals  = [amount, hour_of_day, days_since_last_tx, *cat_vec]
    row       = np.array([raw_vals])
    score     = rf.predict_proba(row)[0][1]

    # Decision thresholds — tunable based on business risk appetite
    if score >= 0.70:
        decision = "BLOCK"
    elif score >= 0.40:
        decision = "FLAG FOR REVIEW"
    else:
        decision = "PASS"

    # Signals come from the MODEL via SHAP — not from hand-written rules.
    # Each feature's contribution toward the fraud class; keep the ones that pushed
    # the score up. For the one-hot category columns, only the ACTIVE one (value 1)
    # is meaningful, so we skip inactive category columns (value 0).
    contributions = explainer.shap_values(row)[0, :, 1]   # fraud-class, sample 0
    ranked = sorted(
        zip(FEATURES, raw_vals, contributions),
        key=lambda t: t[2], reverse=True,
    )
    signals = [
        f"{describe_feature(feat, val)} (contribution +{contrib:.2f})"
        for feat, val, contrib in ranked
        if contrib > 0.02 and not (feat.startswith("merchant_") and val == 0)
    ]

    return {
        "fraud_score": round(score, 3),
        "decision":    decision,
        "top_signals": signals or ["no features pushed toward fraud"],
    }

# Test it directly before involving the agent
print("Direct tool test — clear fraud:")
print(predict_fraud.invoke({"amount": 4200, "merchant_category": "electronics",
                             "hour_of_day": 3, "days_since_last_tx": 47}))
print()
print("Direct tool test — clearly legit:")
print(predict_fraud.invoke({"amount": 42,   "merchant_category": "grocery",
                             "hour_of_day": 14, "days_since_last_tx": 2}))

## Transaction catalog

A set of pre-generated transactions the agent can look up by ID.  
The catalog has a deliberate mix: clearly legit, clearly fraudulent, and borderline cases.

In [ ]:
# Transaction catalog — realistic mix of risk levels
# Presenter note: #2 and #4 should BLOCK; #8 and #10 should FLAG; rest should PASS
CATALOG = {
    1:  {"amount":   42.50, "merchant_category": "grocery",      "hour_of_day": 14, "days_since_last_tx":  2, "merchant_name": "SuperU Lugano"},
    2:  {"amount": 4200.00, "merchant_category": "electronics",  "hour_of_day":  3, "days_since_last_tx": 47, "merchant_name": "TechZone Online"},
    3:  {"amount":   89.99, "merchant_category": "restaurant",   "hour_of_day": 20, "days_since_last_tx":  5, "merchant_name": "Da Mario Ristorante"},
    4:  {"amount": 2800.00, "merchant_category": "online",       "hour_of_day":  2, "days_since_last_tx": 65, "merchant_name": "Unknown Merchant"},
    5:  {"amount":   15.00, "merchant_category": "atm",          "hour_of_day": 11, "days_since_last_tx":  1, "merchant_name": "ATM Via Cattedrale"},
    6:  {"amount":  650.00, "merchant_category": "travel",       "hour_of_day": 19, "days_since_last_tx": 18, "merchant_name": "Swiss Airlines"},
    7:  {"amount":  920.00, "merchant_category": "electronics",  "hour_of_day": 23, "days_since_last_tx": 14, "merchant_name": "MediaMarkt Bellinzona"},
    8:  {"amount": 9800.00, "merchant_category": "online",       "hour_of_day":  4, "days_since_last_tx": 90, "merchant_name": "Unknown Merchant"},
    9:  {"amount":   67.00, "merchant_category": "grocery",      "hour_of_day":  9, "days_since_last_tx":  3, "merchant_name": "Coop Mendrisiotto"},
    10: {"amount": 3100.00, "merchant_category": "electronics",  "hour_of_day": 13, "days_since_last_tx":  7, "merchant_name": "Apple Store Zurich"},
}

import pandas as pd
df_catalog = pd.DataFrame(CATALOG).T
df_catalog.index.name = "ID"
print("Transaction catalog:")
print(df_catalog.to_string())

In [ ]:
@tool
def get_transaction_details(transaction_id: int) -> dict:
    """
    Retrieve the details of a transaction by its ID.

    Args:
        transaction_id: The transaction ID to look up (1–10).
    """
    if transaction_id not in CATALOG:
        return {"error": f"Transaction {transaction_id} not found. Valid IDs: {list(CATALOG.keys())}"}

    tx = CATALOG[transaction_id].copy()
    tx["transaction_id"] = transaction_id
    return tx

# Test it
print(get_transaction_details.invoke({"transaction_id": 2}))

## Build the full agent

We carry over the tools from Block 1 and swap in the real fraud predictor.  
The agent loop, the prompt, the `AgentExecutor` — all identical. Only the tool content changed.

In [ ]:
# ── Tools from Block 1 (redefined here so the block is self-contained) ───────
@tool
def get_current_time(timezone: str) -> str:
    """Get the current date and time in a given IANA timezone.
    Args: timezone: e.g. 'Europe/Zurich' or 'Asia/Tokyo'."""
    try:
        return datetime.now(ZoneInfo(timezone)).strftime("%H:%M on %A %d %B %Y (%Z)")
    except Exception:
        return f"Unknown timezone '{timezone}'."

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Use Python syntax e.g. '4200 * 0.12'.
    Args: expression: a valid Python math expression."""
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    try:
        return str(round(float(eval(expression, {"__builtins__": {}}, allowed)), 4))
    except Exception as e:
        return f"Error: {e}"

# ── Full toolkit ──────────────────────────────────────────────────────────────
all_tools = [
    get_transaction_details,   # new in Block 3
    predict_fraud,             # now real — was a stub in Block 1
    get_current_time,          # carried over from Block 1
    calculator,                # carried over from Block 1
]

agent    = create_tool_calling_agent(llm, all_tools, prompt)
executor = AgentExecutor(agent=agent, tools=all_tools, verbose=True)

print(f"Agent ready with {len(all_tools)} tools:")
for t in all_tools:
    print(f"  • {t.name}")

## 💥 The demo

Same question format as Block 1. Different result.

In [ ]:
ask(executor,
    "Transaction #2 just triggered an alert. "
    "Look it up, assess the fraud risk, and tell me whether we should block it.")

Now a borderline case — watch the agent reason through the ambiguity:

In [ ]:
ask(executor,
    "We have two flagged transactions: #8 and #10. "
    "Check both, compare their risk profiles, and recommend which one to prioritise.")

## 🎮 Your turn

Pick any transaction from the catalog, or describe one of your own — the agent will handle both.

**Ideas:**
- *"Check transactions #2 and #4 — are they likely the same attacker?"*
- *"Transaction just came in: €500 at an online store at 11pm, last transaction was 3 days ago. Safe?"*
- *"Compare #7 and #10 — both electronics, different risk. Why?"*

In [ ]:
# Change and re-run
your_question = "Check transaction #7 and explain why it scores lower than #2 despite being electronics at night."

ask(executor, your_question)

## ✅ What just happened

```
Block 1                          Block 3
───────────────────────          ─────────────────────────────────
predict_fraud() stub        →    predict_fraud() ← Random Forest
  always returns 0.87              real score from trained model
  always says BLOCK                decision from data
  hardcoded signals                signals from SHAP — the model's
                                   own reasoning, per transaction
                             
AgentExecutor ──────────────────────────────────── unchanged
prompt ─────────────────────────────────────────── unchanged  
get_current_time ───────────────────────────────── unchanged
calculator ─────────────────────────────────────── unchanged
```

One function replaced. Everything else stayed the same.

This is how production AI systems are maintained: tools evolve independently of the agent that calls them.  
The ML team ships a better model → the agent immediately becomes more capable — no changes to the agent code.

And because the signals come from SHAP rather than hand-written rules, the agent's explanation always reflects what the model genuinely weighted — even as the model is retrained.

---
## ⚡ Bonus 1 — A second model as a competing tool ⭐⭐⭐

Add a **Logistic Regression** as a second fraud tool. The agent can now call both, compare their scores, and reason about disagreements.

This reflects a real pattern in production: ensemble approaches where multiple models vote, or where a fast/interpretable model runs first and a heavier model only kicks in for borderline cases.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── Retrain on the same synthetic data (same seed = same split) ────────────
np.random.seed(42)
N_ = 15_000
CATS_ = ["grocery","online","electronics","restaurant","travel","atm"]
amt_  = np.random.lognormal(4.5,1.2,N_).clip(1,20_000).round(2)
hr_   = np.random.randint(0,24,N_)
days_ = np.clip(np.random.exponential(10,N_).astype(int),0,120)
cat_  = np.random.choice(CATS_,N_,p=[.35,.20,.15,.15,.10,.05])
risk_ = (0.40*(amt_>1500)+0.30*((hr_<=4)|(hr_>=23))+0.20*(days_>25)+
         0.10*np.isin(cat_,["electronics","online"])+np.random.normal(0,.12,N_)).clip(0,1)
y_    = (risk_ >= np.percentile(risk_,95)).astype(int)

# One-hot encode with the SAME encoder loaded from Block 2 — identical columns
cat_oh = ohe.transform(pd.DataFrame({"merchant_category": cat_}))
X_     = np.column_stack([amt_, hr_, days_, cat_oh])
X_tr, X_te, y_tr, y_te = train_test_split(X_, y_, test_size=0.2, stratify=y_, random_state=42)

lr = Pipeline([("scaler", StandardScaler()),
               ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
lr.fit(X_tr, y_tr)
print("Logistic Regression trained.")

from sklearn.metrics import classification_report
print(classification_report(y_te, lr.predict(X_te), target_names=["Legit","Fraud"]))

In [ ]:
@tool
def predict_fraud_lr(amount: float, merchant_category: str,
                     hour_of_day: int, days_since_last_tx: int) -> dict:
    """
    Alternative fraud predictor using Logistic Regression — faster and more interpretable
    than the Random Forest, but potentially less accurate on complex patterns.
    Use alongside predict_fraud to compare scores.

    Args:
        amount:             Transaction amount in EUR.
        merchant_category:  Merchant category (grocery, online, electronics, restaurant, travel, atm).
        hour_of_day:        Hour of the transaction (0-23).
        days_since_last_tx: Days since the customer last transacted.
    """
    if merchant_category not in CATEGORIES:
        return {"error": f"Unknown category. Valid: {CATEGORIES}"}
    cat_vec  = encode_category(merchant_category)
    features = [[amount, hour_of_day, days_since_last_tx, *cat_vec]]
    score    = lr.predict_proba(features)[0][1]
    decision = "BLOCK" if score >= 0.70 else ("FLAG FOR REVIEW" if score >= 0.40 else "PASS")
    return {"fraud_score": round(score, 3), "decision": decision, "model": "Logistic Regression"}

# ── New agent with both models ────────────────────────────────────────────────
tools_dual    = all_tools + [predict_fraud_lr]
agent_dual    = create_tool_calling_agent(llm, tools_dual, prompt)
executor_dual = AgentExecutor(agent=agent_dual, tools=tools_dual, verbose=True)

ask(executor_dual,
    "Check transaction #10 using both fraud models. "
    "Do they agree? If they disagree, which one would you trust more and why?")

## ⚡ Bonus 2 — Conversation memory ⭐⭐⭐

Right now every `ask()` call starts fresh — the agent has no memory of previous turns.

Add `chat_history` so the agent can reason **across multiple questions in sequence**.  
This turns a one-shot tool into something closer to a real analyst session.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Updated prompt with a chat_history slot
prompt_memory = ChatPromptTemplate.from_messages([
    ("system",
     "You are a fraud analyst assistant at a bank. "
     "Use your tools to investigate transactions and give clear, reasoned recommendations. "
     "Always cite the fraud score and the signals that drove it. "
     "Remember context from earlier in the conversation."),
    MessagesPlaceholder("chat_history"),    # ← new
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent_mem    = create_tool_calling_agent(llm, all_tools, prompt_memory)
executor_mem = AgentExecutor(agent=agent_mem, tools=all_tools, verbose=False)
# verbose=False here so the multi-turn exchange is easier to read

chat_history = []   # grows with each turn

def ask_memory(question):
    """Run one turn of the agent, persisting the conversation history."""
    result = executor_mem.invoke({"input": question, "chat_history": chat_history})
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result["output"]))
    print(f"\nYou: {question}")
    print(f"Agent: {result['output']}")
    return result["output"]

# Multi-turn demo — the second question only makes sense given the first
ask_memory("Check transactions #2 and #4. What are their fraud scores?")
ask_memory("Both were submitted from the same IP address 10 minutes apart. Does that change your assessment?")
ask_memory("What would you recommend we do about the customer's account?")

---
The agent is fully wired.

Move on to **Block 4 →** — recap and next steps.


<br>

---

# Part 4 · Recap 🎯

---

## The journey

```
  Block 1                Block 2                 Block 3
  ─────────              ─────────               ─────────
  Built an agent    →    Trained a real     →    Replaced the stub
  with simple            fraud model on          with the real model
  tools (clock,          real + synthetic        — agent now reasons
  calculator) and        data, learned why       about live fraud
  a fraud STUB           accuracy lies           risk in plain English

       │                      │                       │
       ▼                      ▼                       ▼
  "An agent is just     "Accuracy is the        "Swap the tool, keep
   an LLM that can        wrong metric for         the agent — that is
   call functions"        rare events"             how production works"
```

Three ideas, one coherent system.

## The one thing to remember

> **The agent loop never changes. Tools are what make it powerful.**

A clock, a calculator, a fraud model — to the agent they are all the same kind of thing: a capability it can choose to use. You extend what the agent can do by adding tools, not by rewriting its logic.

This means an AI agent is not a monolith you build once. It is a thin reasoning layer on top of capabilities your teams already have — models, databases, APIs, internal services. The hard work your organisation has already done becomes the toolkit.

## Your takeaway — the whole pattern in 20 lines

Everything we built reduces to this. Copy it, replace the tool with something from your own work, and you have a starting point.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

# 1. Define a capability — anything your team can do in Python
@tool
def my_tool(some_input: str) -> str:
    """Describe what this does — the model reads this to decide when to call it.
    Args: some_input: describe the argument."""
    return f"processed: {some_input}"

# 2. Set up the model and prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# 3. Wire it together and run
agent    = create_tool_calling_agent(llm, [my_tool], prompt)
executor = AgentExecutor(agent=agent, tools=[my_tool], verbose=True)

executor.invoke({"input": "Use the tool on the word 'hello'."})

## Where this applies in your work

> 🗣️ **Open question for the room: where could this pattern fit in your team?**

Three concrete directions, each just a different tool behind the same agent:

**1. AML transaction monitoring**  
Swap the fraud model for an anti-money-laundering model. Add tools to pull customer KYC data and transaction history. The agent flags suspicious patterns, explains why, and drafts the first version of a Suspicious Activity Report for a human to review.

**2. Loan pre-screening assistant**  
Put a credit risk model behind the tool. The agent gathers applicant details, calls the model, and explains the decision in plain language — applying the same criteria every time, with an auditable reasoning trail.

**3. Regulatory Q&A assistant**  
Give the agent a tool to query live risk indicators plus a retrieval tool over your policy documents. Staff ask compliance questions in plain English and get answers grounded in current data and current rules.

## Before you deploy this for real

A workshop demo and a production banking system are very different things. If you take this further, the hard parts are not the code:

- **Human in the loop** — for consequential decisions (blocking a payment, denying a loan), the agent should recommend, not decide. A person owns the final call.
- **Model governance** — a model that affects customers needs validation, monitoring, and documentation. Most banks have a model risk framework this must fit into.
- **Explainability is not optional** — in many jurisdictions you must be able to explain an automated decision to the customer. This is exactly why we used SHAP rather than a black-box score.
- **Bias and fairness** — a model is only as fair as its training data. Synthetic data hid this; real customer data will not.
- **The LLM can be wrong** — the agent's reasoning is persuasive but not infallible. The tools provide ground truth; the LLM narrates. Keep that boundary clear.

## To go further

| Topic | Where to look |
|---|---|
| Agents & tools | LangChain documentation — "Agents" and "Tools" guides |
| Function calling | OpenAI documentation — "Function calling" |
| The ML side | scikit-learn user guide — classification, model evaluation |
| Explainability | SHAP documentation and the original SHAP paper |
| Imbalanced data | `imbalanced-learn` library (SMOTE and related techniques) |

The full notebooks from today are yours to keep, extend, and adapt.

---

## That's a wrap 🎉

You built an AI agent, trained a fraud model, and combined them into a system that reasons about real transactions — in two hours, from scratch.

The pattern you learned scales far beyond fraud. Any model your organisation has, any data source, any internal service — they can all become tools an agent reasons with.

Now go build something.